In [10]:
import pandas as pd
# On recharge le fichier CSV généré à l'étape précédente
df = pd.read_csv("churn_data.csv")
df.head() # affiche les 5 premières lignes, pour un contrôle visuel rapide

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [11]:
df.shape # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

(500, 6)

In [12]:
df.describe()  # count (500 clients), moyenne, min, max, écart-type, 25%, 50%, 75% des clients

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [13]:
df.isnull().sum()  # vérifie que les données sont complètes, aucune donnée n'est vide, doit être 0 partout

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [14]:
df["churn"].value_counts() # compte les clients "0" (restent) vs "1" (partent)

churn
0    481
1     19
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import train_test_split

# 1. Conversion de la colonne "abonnement" (texte -> nombres)
df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

# 2. Variables explicatives (features)
X = df[["age", "salaire", "abonnement", "anciennete", "consommation"]]

# 3. Variable cible (target)
y = df["churn"]

# 4. Séparation Train / Test : 80% entraînement, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,     # 20% des données réservées au test
    random_state=42    # reproductibilité, comme pour le dataset
)

In [16]:
X_train.shape

(400, 5)

In [17]:
X_test.shape

(100, 5)

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# On crée le modèle : une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre (évite qu'il "apprenne par cœur").
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# .fit() déclenche l'APPRENTISSAGE : le modèle analyse X_train et y_train
# pour trouver une relation entre les deux.
model.fit(X_train, y_train)

# On demande au modèle de prédire sur les données de test (jamais vues avant).
y_pred = model.predict(X_test)

# accuracy_score compare les prédictions (y_pred) à la vérité (y_test)
# et renvoie le pourcentage de bonnes réponses.
print("Accuracy :", accuracy_score(y_test, y_pred))

# classification_report détaille la précision et le rappel, classe par classe.
print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [20]:
import joblib
# joblib.dump() sérialise (transforme en fichier binaire) l'objet "model" entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [21]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
n_estimators=50, # moins d'arbres qu'à l'essai 1 (100 -> 50)
max_depth=3, # arbres moins profonds qu'à l'essai 1 (5 -> 3)
random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ---------- ESSAI 3 : on retire la colonne "salaire" ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]

# Séparation Train / Test : 80% entraînement, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entraînement du modèle sur les 4 colonnes restantes
model3 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model3.fit(X_train, y_train)

# Prédiction et évaluation sur les données de test
y_pred3 = model3.predict(X_test)
print("Accuracy :", accuracy_score(y_test, y_pred3))
print(classification_report(y_test, y_pred3))

Accuracy : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        95
           1       1.00      1.00      1.00         5

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [24]:
# ---------- Vérification : que contient VRAIMENT model.pkl maintenant ? ----------
loaded = joblib.load("model.pkl")
# n_features_in_ indique combien de colonnes ce modèle attend en entrée :
# 5 -> c'est l'Essai 1 ou 2 ; 4 -> c'est forcément l'Essai 3 (sans salaire)
print("Nombre de features attendu :", loaded.n_features_in_)

Nombre de features attendu : 5
